In [1]:
import uuid
from langgraph.store.memory import InMemoryStore
from langchain_ollama import OllamaEmbeddings

In [2]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")

In [3]:
vec = embeddings.embed_query("hello")

In [4]:
print(len(vec))

1024


In [5]:
store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1024,                              # Embedding dimensions
        "fields": ["food_preference", "$"]              # Fields to embed
    }
)

In [6]:
user_id = "1"
namespace_for_memory = (user_id, "memories")

store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {"food_preference": "I love pizza"}
)


In [7]:
store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {"drink_preference": "I want to drink water"}
)

In [8]:
results = store.search(
    namespace_for_memory,
    query="what does user like to drink",
    limit=2
)

In [9]:
for rs in results:
    print(rs)
    print('\n-----')

Item(namespace=['1', 'memories'], key='c7120ec0-2152-4471-b9be-4bfce702545f', value={'drink_preference': 'I want to drink water'}, created_at='2026-02-22T10:19:01.085455+00:00', updated_at='2026-02-22T10:19:01.085461+00:00', score=0.7767355002235494)

-----
Item(namespace=['1', 'memories'], key='71813171-3948-471e-9bc5-6fbd1b23d526', value={'food_preference': 'I love pizza'}, created_at='2026-02-22T10:18:59.976349+00:00', updated_at='2026-02-22T10:18:59.976354+00:00', score=0.5911304117110793)

-----


In [10]:
store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {
        "food_preference": "I love Italian cuisine",
        "context": "Discussing dinner plans"
    },
    index=["food_preference"]  # Only embed "food_preferences" field
)